<a href="https://colab.research.google.com/github/Imran0324/Ml-Internship-Assignment/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imran0324/Ml-Internship-Assignment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I am using a **Random Forest Classifier** to predict the `is_declining_label`. My lane is "quick win optimization", which I've framed as a ranking task: finding the pages most likely to decline in traffic so we can refresh them first.
A Random Forest fits because it handles the mix of numerical (search volume, rates) and categorical (tiers, intent) features natively, captures non-linear relationships, and outputs a predicted probability that we can use to rank candidates.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I am using a **Grouped Split by `client_id`** (80% train, 20% test).
This is honest because a single client's pages share underlying domain authority, seasonality, or technical SEO setups. If we randomly shuffle rows, the model could "memorize" a client's performance in the training set and artificially boost test scores. Grouping ensures the model is tested on clients it has never seen before, proving it learns generalizable patterns.


In [1]:
!git clone https://github.com/Imran0324/Ml-Internship-Assignment.git

Cloning into 'Ml-Internship-Assignment'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 145 (delta 52), reused 100 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 1.88 MiB | 4.20 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# Fallback path if running in Colab
local_raw = "../../data/raw/content_refresh_anonymized.csv"
colab_raw = "/content/Ml-Internship-Assignment/data/raw/content_refresh_anonymized.csv"
github_raw = "https://raw.githubusercontent.com/Imran0324/Ml-Internship-Assignment/main/data/raw/content_refresh_anonymized.csv"

if os.path.exists(local_raw):
    df = pd.read_csv(local_raw)
elif os.path.exists(colab_raw):
    df = pd.read_csv(colab_raw)
else:
    df = pd.read_csv(github_raw)

# Basic feature engineering (instead of relying on external scripts)
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
df['log_impressions_90d'] = np.log1p(df['impressions_90d'].fillna(0))
df['log_clicks_90d'] = np.log1p(df['clicks_90d'].fillna(0))
df['log_sessions_90d'] = np.log1p(df['sessions_90d'].fillna(0))
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'].fillna(0))
df['days_with_impressions'] = df['days_with_impressions'].fillna(0)
df['days_with_sessions'] = df['days_with_sessions'].fillna(0)
df['content_age_days'] = df['content_age_days'].fillna(0)
df['days_since_last_update'] = df['days_since_last_update'].fillna(0)
df['ctr'] = df['ctr'].fillna(0)
df['avg_position'] = df['avg_position'].fillna(0)
df['engagement_rate'] = df['engagement_rate'].fillna(0)
df['scroll_rate'] = df['scroll_rate'].fillna(0)
df['ai_traffic_pct'] = df['ai_traffic_pct'].fillna(0)
df['search_volume'] = df['search_volume'].fillna(0)
df['competition'] = df['competition'].fillna(0)
df['cpc'] = df['cpc'].fillna(0)
df['word_count'] = df['word_count'].fillna(0)
df['char_count'] = df['char_count'].fillna(0)

# Filter like the prepare script
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

# Create the grouped split
splitter = GroupShuffleSplit(test_size=0.20, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df['client_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Train size: {len(train_df)} rows, {train_df['client_id'].nunique()} clients")
print(f"Test size: {len(test_df)} rows, {test_df['client_id'].nunique()} clients")


Train size: 23837 rows, 25 clients
Test size: 6163 rows, 7 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Metric: **Precision@100** on predicting `is_declining_label == 1`.
I calculate the baseline score on the test set using the rule from Week 4 (`(11 <= pos <= 20) * (impressions >= 1000) * (ctr < 1.5) * impressions`), rank the items, and measure its precision at the top 100. Then I train the Random Forest, rank the test set by its predicted probabilities, and measure its precision at 100.


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

def precision_at_k(y_true, scores, k: int) -> float:
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty: return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0

features = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
target = "is_declining_label"

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

# Build preprocessing and model pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), MODEL_NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), MODEL_CATEGORICAL_FEATURES)
    ]
)

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1))
])

# Train model
rf_model.fit(X_train, y_train)
test_df['rf_prob'] = rf_model.predict_proba(X_test)[:, 1]

# Calculate baseline score on test set
pos_match = ((test_df['avg_position'] > 10) & (test_df['avg_position'] <= 20)).astype(int)
vol_match = (test_df['impressions_90d'] >= 1000).astype(int)
ctr_match = (test_df['ctr'] < 1.5).astype(int)
test_df['baseline_score'] = pos_match * vol_match * ctr_match * test_df['impressions_90d']

# Evaluate Precision@100
base_rate = y_test.mean()
baseline_p100 = precision_at_k(test_df[target], test_df['baseline_score'], k=100)
model_p100 = precision_at_k(test_df[target], test_df['rf_prob'], k=100)

comparison_df = pd.DataFrame({
    "Method": ["Base Rate (Random)", "Week-4 Baseline Rule", "Random Forest Model"],
    "Precision@100": [f"{base_rate*100:.1f}%", f"{baseline_p100*100:.1f}%", f"{model_p100*100:.1f}%"]
})
display(comparison_df)


,Method,Precision@100
0,Base Rate (Random),51.1%
1,Week-4 Baseline Rule,39.0%
2,Random Forest Model,57.0%


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Top features driving the model:
1. `days_with_impressions` and `log_impressions_90d` (the overall footprint and consistency of traffic)
2. `avg_position`
3. `content_age_days` and `word_count`

Where the model is wrong (False Positives - high probability of decline, but actually stable/up):
- **Stale Keyword Articles**: The top false positives are all "keyword articles" that haven't been updated in over 3 months (~103 days), yet they still maintain decent traffic (300-1500 impressions). The model strongly associates staleness in keyword articles with decay, but misses that some true evergreen topics can coast for much longer without a refresh.



In [4]:
# Feature Importances
rf_classifier = rf_model.named_steps["classifier"]
cat_encoder = rf_model.named_steps["preprocessor"].named_transformers_["cat"]
encoded_cats = cat_encoder.get_feature_names_out(MODEL_CATEGORICAL_FEATURES)
all_feature_names = MODEL_NUMERIC_FEATURES + list(encoded_cats)

importances = rf_classifier.feature_importances_
imp_df = pd.DataFrame({"Feature": all_feature_names, "Importance": importances})
imp_df = imp_df.sort_values("Importance", ascending=False).head(5)
print("--- Top 5 Features ---")
print(imp_df.to_string(index=False))

# False Positive Examples (Top ranked by model, but label is 0)
print("\n--- Top False Positives (Predicted to decline, but didn't) ---")
fps = test_df[(test_df[target] == 0)].sort_values("rf_prob", ascending=False).head(3)
display_cols = ['content_id', 'rf_prob', 'impressions_90d', 'days_since_last_update', 'content_type', target]
display(fps[display_cols])


--- Top 5 Features ---
              Feature  Importance
days_with_impressions    0.151561
  log_impressions_90d    0.120493
         avg_position    0.099563
     content_age_days    0.092945
           word_count    0.057422

--- Top False Positives (Predicted to decline, but didn't) ---


,content_id,rf_prob,impressions_90d,days_since_last_update,content_type,is_declining_label
10080,content_35d63627bf3e,0.851181,1525,103,keyword article,0
5011,content_c148e44db30d,0.845996,335,104,keyword article,0
11061,content_0b47dae0c7f9,0.844540,1191,103,keyword article,0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
